# Fine-tuning Qwen2.5-7B-Instruct (QLoRA / Unsloth) — Pipeline Planner

Fine-tunes the planner agent to emit the unified **ADF + Databricks** pipeline-config
JSON from a CSV schema + a natural-language request.

**Target hardware:** Intel CPU · **RTX 3080 Ti (12 GB VRAM)** · 64 GB RAM.
Qwen2.5-7B in 4-bit QLoRA uses ~7-9 GB VRAM at `max_seq_length=2048`, so it fits.

---
## 0. Prerequisites (READ FIRST)

**Operating system — strongly recommended: Linux or Windows + WSL2 (Ubuntu).**
Unsloth + Triton + bitsandbytes are most reliable there. Native Windows works but is
fragile; if you must, see the *Native Windows* note in the install cell.

1. **NVIDIA driver + CUDA**: driver supporting CUDA 12.1+ (`nvidia-smi` should work).
2. **Python 3.11** (3.10 also fine; avoid 3.12+ for Unsloth).
   Use a fresh conda/venv environment:
   ```bash
   conda create -n qwenft python=3.11 -y && conda activate qwenft
   ```
3. **~30 GB free disk** (base model download ~5 GB + checkpoints).
4. Copy **`synthetic_planner_dataset.json`** next to this notebook.
5. (Optional) a Hugging Face account/token if the model needs auth (Qwen is public).

> **Dataset size warning:** 20 records is enough to smoke-test the pipeline but NOT to
> produce a good model. Regenerate a larger set first (e.g. change `range(20)` ->
> `range(2000)` in `generate_dataset.py`) and aim for **a few thousand** examples.

## 1. Environment check
Confirm the GPU is visible before installing anything heavy.

In [ ]:
import subprocess, sys
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi']).decode())
except Exception as e:
    print('nvidia-smi not found — check your NVIDIA driver install:', e)

## 2. Install dependencies

Run this cell **once** per environment. It installs a CUDA-12.1 PyTorch build,
Unsloth, and the training stack. Unsloth pins compatible `transformers`/`trl`/`peft`.

**Native Windows note:** if you are NOT in WSL2/Linux, also run:
`pip install triton-windows` and ensure `bitsandbytes>=0.43` (has Windows wheels).
If `xformers`/`flash-attn` fail to build, ignore them — Unsloth ships its own kernels.

In [ ]:
# 1) PyTorch (CUDA 12.1). Skip if a CUDA torch is already installed.
#    Check first:  python -c "import torch;print(torch.__version__,torch.cuda.is_available())"
%pip install --quiet 'torch==2.4.1' --index-url https://download.pytorch.org/whl/cu121

# 2) Unsloth (official one-liner). It pulls a COHERENT, mutually-compatible
#    transformers / trl / peft / accelerate / bitsandbytes / datasets set,
#    so do NOT pin those yourself.
%pip install --quiet --upgrade --no-cache-dir unsloth

# Native Windows only (NOT WSL2/Linux), uncomment:
# %pip install --quiet triton-windows
print('Install step done. RESTART THE KERNEL before continuing.')

## 3. Configuration

In [ ]:
DATASET_PATH    = 'synthetic_planner_dataset.json'
BASE_MODEL      = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'  # pre-quantized 4-bit
MAX_SEQ_LENGTH  = 2048      # raise to 4096 if the token-stats cell says so
LORA_RANK       = 16
LORA_ALPHA      = 16
OUTPUT_LORA     = 'qwen_planner_lora'
OUTPUT_MERGED   = 'qwen_planner_merged'
SEED            = 3407

## 4. Build the chat dataset
Each record `{schema, user_prompt, config}` becomes a 3-turn chat:
system (task + DSL grammar) -> user (schema + request) -> assistant (config JSON).
The system prompt mirrors the real `groq_planner` contract so this model can be
swapped in directly.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = "You are a hybrid Azure Data Factory + Databricks pipeline architect. ADF orchestrates (control plane); Databricks computes (execution plane). Given a CSV schema and a user request, output exactly ONE JSON object describing a multi-stage medallion pipeline (raw -> bronze -> silver -> ...).\n\nStage types:\n  - 'copy'    : ADF Copy Activity, ingestion ONLY (stage0 -> stage1), no transforms.\n  - 'notebook': Databricks PySpark; does transformations, filter, aggregation.\n\nTransformation DSL ('output_col = expr'): upper/lower/trim/initCap/length, toInteger/toDouble/toString/toTimestamp, concat/substring/regexReplace, year/month/dayOfMonth, currentTimestamp(), and arithmetic (+ - * /). ALWAYS add 'processed_time = currentTimestamp()' to every notebook stage.\n\nFilter ('filter_condition'): equals/notEquals/greater/less(toInteger(col), n), equals(col, 'value'), isNull(col), or 'col > n'.\n\nAggregation (optional, notebook stages only): {\"group_by\": [...], \"aggregations\": [{\"op\": avg|sum|min|max|count, \"column\": <col or '*'>, \"alias\": <name>}]}. avg/sum require numeric columns; group_by and aggregated columns MUST exist in the schema.\n\nRules: first stage is 'copy', the rest are 'notebook'; reference ONLY columns that exist in the schema; preserve column names exactly. Output ONLY the JSON object \u2014 no markdown, no commentary."

def build_user_content(rec):
    s, cfg = rec['schema'], rec['config']
    return (
        f"CSV Columns: {s['columns']}\n"
        f"Column Types: {s['inferred_types']}\n"
        f"Row Count: {s['row_count']}\n"
        f"File Size: {s['size_hint']}\n\n"
        f"Sample Data:\n{json.dumps(s['samples'][:3], indent=2)}\n\n"
        f'User Prompt: "{rec["user_prompt"]}"\n\n'
        f"Number of stages: {cfg['num_containers']}\n"
        f"Container names: {cfg['containers_to_create']}\n\n"
        'Design the complete unified ADF+Databricks pipeline configuration JSON:'
    )

def record_to_messages(rec):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_content(rec)},
        {'role': 'assistant', 'content': json.dumps(rec['config'], indent=2)},
    ]

with open(DATASET_PATH, encoding='utf-8') as f:
    raw = json.load(f)
print(f'Loaded {len(raw)} records')

examples = [{'messages': record_to_messages(r)} for r in raw]
ds = Dataset.from_list(examples)

# Train/val split (guard tiny datasets).
if len(ds) >= 20:
    split = ds.train_test_split(test_size=0.1, seed=SEED)
    train_ds, eval_ds = split['train'], split['test']
else:
    train_ds, eval_ds = ds, None
print('train:', len(train_ds), '| eval:', len(eval_ds) if eval_ds else 0)

## 5. Load model + tokenizer (4-bit) and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # auto (bf16 on Ampere)
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state   = SEED,
)

## 6. Token-length stats
Confirms `MAX_SEQ_LENGTH` is large enough (no silent truncation of targets).

In [ ]:
lens = [len(tokenizer.apply_chat_template(e['messages'], tokenize=True))
        for e in examples]
print('max tokens:', max(lens), '| mean:', sum(lens)//len(lens))
assert max(lens) <= MAX_SEQ_LENGTH, (
    f'Increase MAX_SEQ_LENGTH to >= {max(lens)} and reload the model')
print('OK: all examples fit in MAX_SEQ_LENGTH =', MAX_SEQ_LENGTH)

## 7. Apply Qwen chat template -> training text

In [ ]:
def formatting_func(batch):
    return {'text': [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch['messages']
    ]}

train_fmt = train_ds.map(formatting_func, batched=True, remove_columns=train_ds.column_names)
eval_fmt  = (eval_ds.map(formatting_func, batched=True, remove_columns=eval_ds.column_names)
             if eval_ds else None)
print(train_fmt[0]['text'][:600])

## 8. Trainer
Loss is computed on the **assistant response only** (`train_on_responses_only`),
so the model is not trained to reproduce the prompt.

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Current TRL puts all training + dataset args inside SFTConfig.
sft_args = SFTConfig(
    dataset_text_field          = 'text',
    max_seq_length              = MAX_SEQ_LENGTH,
    packing                     = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,    # effective batch 8
    warmup_ratio                = 0.05,
    num_train_epochs            = 3,    # small data -> a few epochs
    learning_rate               = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps               = 1,
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    seed                        = SEED,
    output_dir                  = 'outputs',
    report_to                   = 'none',
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,    # Unsloth accepts this alias for processing_class
    train_dataset = train_fmt,
    eval_dataset  = eval_fmt,
    args          = sft_args,
)

# Mask everything except the assistant turn (Qwen ChatML markers),
# so loss is computed on the config JSON only.
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part    = '<|im_start|>assistant\n',
)

## 9. Train

In [ ]:
stats = trainer.train()
print(stats)

## 10. Quick inference test
Generate a config for a held-out-style prompt and check it parses as JSON.

In [ ]:
FastLanguageModel.for_inference(model)

test_user = build_user_content({
    'schema': {
        'columns': ['order_id','region','product','quantity','price','customer_email'],
        'inferred_types': {'order_id':'integer','region':'string','product':'string',
            'quantity':'integer','price':'double','customer_email':'string'},
        'row_count': 5000, 'size_hint': 'small (< 5MB)',
        'samples': [{'order_id':'101','region':'US','product':'Mouse',
            'quantity':'3','price':'19.99','customer_email':'a@b.com'}],
    },
    'user_prompt': 'keep orders over 100 and average price per region',
    'config': {'num_containers': 3, 'containers_to_create': ['raw','bronze','silver']},
})

msgs = [{'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':test_user}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                       return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.2,
                     top_p=0.8, do_sample=True)
text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(text)
import json as _j
try:
    _j.loads(text); print('\nValid JSON ✔')
except Exception as e:
    print('\nNOT valid JSON yet (more data/epochs needed):', e)

## 11. Save
LoRA adapters are tiny (~80 MB). The merged 16-bit model is a standalone ~15 GB
checkpoint you can serve with vLLM / transformers.

In [ ]:
# LoRA adapters only
model.save_pretrained(OUTPUT_LORA)
tokenizer.save_pretrained(OUTPUT_LORA)
print('Saved adapters ->', OUTPUT_LORA)

# Optional: merged fp16 (uncomment; needs ~15 GB disk + RAM)
# model.save_pretrained_merged(OUTPUT_MERGED, tokenizer, save_method='merged_16bit')

# Optional: GGUF for llama.cpp / Ollama (uncomment; builds llama.cpp, slow)
# model.save_pretrained_gguf('qwen_planner_gguf', tokenizer, quantization_method='q4_k_m')

## 12. Plug back into the planner

Two options to use the fine-tuned model in `groq_planner`-style code:

1. **Local serve (vLLM, OpenAI-compatible):**
   ```bash
   pip install vllm
   vllm serve qwen_planner_merged --max-model-len 4096
   ```
   Then point the planner at `http://localhost:8000/v1/chat/completions`
   (same request shape as the Groq call; set `model` to the served name).

2. **Adapters + transformers** in-process: load `BASE_MODEL` + `PeftModel.from_pretrained`
   on `OUTPUT_LORA` and call `apply_chat_template` with the SAME `SYSTEM_PROMPT`.

Keep the **SYSTEM_PROMPT identical** at inference to how it was trained.